In [0]:
BASE = "abfss://landing@ticjarvis.dfs.core.windows.net"

users = spark.read.option("header", True).csv(f"{BASE}/users_data.csv")
print("users:", users.count())
display(users.limit(5))

users: 2000


id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
825,53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,$29278,$59696,$127613,787,5
1746,53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,$37891,$77254,$191349,701,5
1718,81,67,1938,11,Female,766 Third Drive,34.02,-117.89,$22681,$33483,$196,698,5
708,63,63,1957,1,Female,3 Madison Street,40.71,-73.99,$163145,$249925,$202328,722,4
1164,43,70,1976,9,Male,9620 Valley Stream Drive,37.76,-122.44,$53797,$109687,$183855,675,1


In [0]:
from pyspark.sql.functions import from_json, explode, col
from pyspark.sql.types import MapType, StringType, StructType, StructField

schema = StructType([StructField("target", MapType(StringType(), StringType()))])

fraud = (spark.read.text(f"{BASE}/train_fraud_labels.json", wholetext=True)
         .select(from_json(col("value"), schema).alias("j"))
         .select(explode("j.target").alias("transaction_id", "is_fraud")))

print("fraud labels:", fraud.count())
display(fraud.limit(5))

fraud labels: 8914963


transaction_id,is_fraud
10649266,No
23410063,No
9316588,No
12478022,No
9558530,No


In [0]:
mcc = (spark.read.text(f"{BASE}/mcc_codes.json", wholetext=True)
       .select(from_json(col("value"), MapType(StringType(), StringType())).alias("m"))
       .select(explode("m").alias("mcc", "mcc_description")))

print("mcc codes:", mcc.count())
display(mcc.limit(5))

mcc codes: 109


mcc,mcc_description
5812,Eating Places and Restaurants
5541,Service Stations
7996,"Amusement Parks, Carnivals, Circuses"
5411,"Grocery Stores, Supermarkets"
4784,Tolls and Bridge Fees


In [0]:
from pyspark.sql.functions import from_json, explode, col, current_timestamp, lit
from pyspark.sql.types import MapType, StringType, StructType, StructField

BASE = "abfss://landing@ticjarvis.dfs.core.windows.net"
CATALOG = spark.sql("SELECT current_catalog()").collect()[0][0]
spark.sql(f"USE CATALOG {CATALOG}")
for s in ["bronze", "silver", "gold"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {s}")

def save(df, name, source):
    (df.withColumn("_ingested_at", current_timestamp())
       .withColumn("_source", lit(source))
       .write.mode("overwrite").option("overwriteSchema", "true")
       .saveAsTable(f"bronze.{name}"))
    print(name, "->", spark.table(f"bronze.{name}").count())

In [0]:
# --- from ADLS external location ---
save(spark.read.option("header", True).csv(f"{BASE}/users_data.csv"),
     "users", "adls/users_data.csv")

save(spark.read.text(f"{BASE}/mcc_codes.json", wholetext=True)
        .select(from_json(col("value"), MapType(StringType(), StringType())).alias("m"))
        .select(explode("m").alias("mcc", "mcc_description")),
     "mcc_codes", "adls/mcc_codes.json")

fschema = StructType([StructField("target", MapType(StringType(), StringType()))])
save(spark.read.text(f"{BASE}/train_fraud_labels.json", wholetext=True)
        .select(from_json(col("value"), fschema).alias("j"))
        .select(explode("j.target").alias("transaction_id", "is_fraud")),
     "fraud_labels", "adls/train_fraud_labels.json")

users -> 2000
mcc_codes -> 109
fraud_labels -> 8914963


In [0]:
# --- cards & transactions from ADLS ---
# NOTE: interim path. JDBC ingestion (see cell 7) blocked by Azure SQL connectivity.

save(spark.read.option("header", True).csv(f"{BASE}/cards_data.csv"),
     "cards", "adls/cards_data.csv")

save(spark.read.option("header", True)
        .option("multiLine", True).option("quote", '"').option("escape", '"')
        .csv(f"{BASE}/transactions_data.csv"),
     "transactions", "adls/transactions_data.csv")

cards -> 6146
transactions -> 13305915


In [0]:
%sql
SELECT 'transactions' t, COUNT(*) n FROM bronze.transactions
UNION ALL SELECT 'cards', COUNT(*) FROM bronze.cards
UNION ALL SELECT 'users', COUNT(*) FROM bronze.users
UNION ALL SELECT 'mcc_codes', COUNT(*) FROM bronze.mcc_codes
UNION ALL SELECT 'fraud_labels', COUNT(*) FROM bronze.fraud_labels

t,n
transactions,13305915
cards,6146
users,2000
mcc_codes,109
fraud_labels,8914963


In [0]:
%sql
SELECT COUNT(*) FROM bronze.transactions WHERE errors LIKE '%,%'

COUNT(*)
941
